# Data Engineering and Management Workshop — Solutions

This workbook contains the worked solution. Use it to check the method, the code, and the reasoning behind each step.

The case study follows a fictional prosthetics charity and shows the core parts of a data engineering workflow: ingestion, cleaning, joins, and privacy masking.

You will work through four themes:

- Ingestion and pipelines
- Data cleaning
- Cybersecurity and privacy masking
- Relational storage and joins

The dataset `prosthetics_data.csv` is in the parent folder.

---
## Step 0 — Setup

---
## Section 1 — Ingestion & Pipelines (ETL vs ELT)

ETL means **Extract, Transform, Load**. Data is validated before it reaches the destination system. In a healthcare setting that usually means privacy checks and validation happen before sensitive records move anywhere.

ELT means **Extract, Load, Transform**. Raw data lands first and is transformed later inside a warehouse. That works well at scale, but raw patient records sit exposed until the transform runs.

This workshop follows the ETL path: clean and mask before anything leaves the secure environment.

In [ ]:

csv_candidates = [
    Path('prosthetics_data.csv'),
    Path('..') / 'prosthetics_data.csv',
]

for candidate in csv_candidates:
    if candidate.exists():
        csv_path = candidate
        break
else:
    raise FileNotFoundError('Could not find prosthetics_data.csv.')

df = pd.read_csv(csv_path)
print(df.head().to_string())
print('\nMissing values per column:')
print(df.isna().sum())
df.info()

---
## Section 2 — Data Cleaning

Three cleaning problems: messy ages, inconsistent date formats, and inconsistent casing on categories.

### 2a — Parse Age_Input

In [ ]:
import re

CURRENT_YEAR = 2026

def parse_age_input(value):
    if pd.isna(value):
        return None

    if isinstance(value, (int, float)) and not pd.isna(value):
        numeric_value = int(value)
    else:
        match = re.search(r'\d+', str(value))
        if not match:
            return None
        numeric_value = int(match.group())

    if numeric_value < 100:
        return numeric_value
    if numeric_value > 1900:
        return CURRENT_YEAR - numeric_value
    return numeric_value

df['Age_Years'] = df['Age_Input'].apply(parse_age_input)
df[['Patient_ID', 'Age_Input', 'Age_Years']]

### 2b — Parse Implant_Date

In [ ]:
def parse_implant_date(raw):
    """Return YYYY-MM-DD string for any of the date formats in this dataset."""
    if pd.isna(raw):
        return None
    s = re.sub(r'(\d+)(st|nd|rd|th)', r'\1', str(raw).strip())
    # Year-first formats (YYYY-MM-DD, YYYY/MM/DD) must be handled before dayfirst
    # parsing: pd.to_datetime with dayfirst=True swaps month and day even on
    # unambiguous ISO strings like 2023-09-01, turning them into 2023-01-09.
    m = re.match(r'^(\d{4})[-/](\d{1,2})[-/](\d{1,2})$', s)
    if m:
        y, mo, d = m.groups()
        return f'{y}-{int(mo):02d}-{int(d):02d}'
    try:
        return pd.to_datetime(s, dayfirst=True).strftime('%Y-%m-%d')
    except Exception:
        return None

df['Implant_Date_Clean'] = df['Implant_Date'].apply(parse_implant_date)
df[['Patient_ID', 'Implant_Date', 'Implant_Date_Clean']]

### 2c — Standardise Amputation_Cause

In [ ]:
df['Amputation_Cause_Clean'] = df['Amputation_Cause'].str.lower().str.strip()
df[['Patient_ID', 'Amputation_Cause', 'Amputation_Cause_Clean']]

---
## Section 3 — Cybersecurity & Cloud Privacy (PII Masking)

PII means Personally Identifiable Information. Names, contact details, and other identifiers should be protected, minimised, or removed before data is shared or uploaded.

Before patient data is moved to public cloud storage or joined with external datasets, it should be anonymised or pseudonymised. Hashing can replace names with stable tokens that do not expose the original value. This is part of the Transform phase in ETL.

In [ ]:
import hashlib

def anonymize_name(value):
    normalized_value = str(value).strip().lower()
    return hashlib.sha256(normalized_value.encode('utf-8')).hexdigest()

df['Full_Name'] = df['Full_Name'].apply(anonymize_name)
df[['Patient_ID', 'Full_Name']]

---
## Section 4 — Relational Storage & Joins

Structured data fits into tables, rows, and columns, so SQL databases are a good fit when you need rules and relationships. Unstructured data includes notes, messages, and scanned documents.

Primary keys identify rows. Foreign keys point to related rows in other tables. Joins use those keys to connect data without duplicating it.

For this charity, patient records can be joined to device inventory so analysts can see which model was fitted, what it cost, and who made it. Privacy-masked identifiers are now safe to join and share.

In [ ]:
device_inventory = {
    'M210': {'Model_Name': 'AeroStep Lite', 'Cost_USD': 950, 'Manufacturer': 'Stride MedTech'},
    'M302': {'Model_Name': 'FlexFoot Alpha', 'Cost_USD': 1200, 'Manufacturer': 'Global Limb Works'},
    'M404': {'Model_Name': 'CrestFit Pro', 'Cost_USD': 1850, 'Manufacturer': 'OrthoSphere'},
    'M505': {'Model_Name': 'AdaptMax Field', 'Cost_USD': 2100, 'Manufacturer': 'Humanity Mobility Systems'},
}

device_inventory_df = pd.DataFrame.from_dict(device_inventory, orient='index').reset_index()
device_inventory_df = device_inventory_df.rename(columns={'index': 'Device_Model_ID'})
device_inventory_df = device_inventory_df[['Device_Model_ID', 'Model_Name', 'Cost_USD', 'Manufacturer']]

patient_device_df = pd.merge(df, device_inventory_df, on='Device_Model_ID', how='inner')
patient_device_df[['Patient_ID', 'Device_Model_ID', 'Model_Name', 'Cost_USD', 'Manufacturer']]

In [ ]:
print(f'Rows before merge: {len(df)}')
print(f'Rows after merge : {len(patient_device_df)}')
assert len(patient_device_df) == 15, 'Row count mismatch — check for unknown Device_Model_IDs'

output_cols = [
    'Patient_ID', 'Full_Name', 'Age_Years',
    'Implant_Date_Clean', 'Amputation_Cause_Clean',
    'Device_Model_ID', 'Model_Name', 'Cost_USD', 'Manufacturer',
]
patient_device_df[output_cols]

## Wrap-Up

By the end of the exercises, you should be able to explain how messy inputs are handled, how joins connect business tables, and why privacy controls matter before cloud upload.